In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import os

from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder

# -------------------- Load Test Data --------------------
test_transaction = pd.read_csv('datasets/test_transaction.csv')
test_identity = pd.read_csv('datasets/test_identity.csv')

# Normalize test_identity column names to match train_identity
test_identity.columns = test_identity.columns.str.replace('-', '_')

# Merge on TransactionID
test_df = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

# Keep a copy of TransactionID for submission
submission_ids = test_df['TransactionID']

# -------------------- Preprocessing --------------------
# Fill missing values
categorical_cols = test_df.select_dtypes(include='object').columns
numerical_cols = test_df.select_dtypes(include=['int64', 'float64']).columns

# Fill NaNs: Categorical with "Missing", Numeric with median
test_df[categorical_cols] = test_df[categorical_cols].fillna("Missing")
for col in numerical_cols:
    median_val = test_df[col].median()
    test_df[col] = test_df[col].fillna(median_val)

# -------------------- Load Training Columns --------------------
# Load and apply encoding/scaling based on full training data
train_data = pd.read_csv('datasets/train_dataset.csv')

# Target and feature separation
y = train_data['isFraud']
X = train_data.drop(columns=['isFraud'])

# Identify categorical and numerical again
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Target Encoding using full training set
encoder = TargetEncoder(cols=categorical_cols)
X[categorical_cols] = encoder.fit_transform(X[categorical_cols], y)
test_df[categorical_cols] = encoder.transform(test_df[categorical_cols])

# Standard Scaling on numeric columns
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

# Align test columns with training
test_df = test_df[X.columns]  # Same column order

# -------------------- Predict Using 10 Fold Models --------------------
preds = np.zeros(len(test_df))

for fold in range(1, 11):
    print(f"Loading model for Fold {fold}")
    model_path = f"models/lightgbm/fold_{fold}.txt"
    model = lgb.Booster(model_file=model_path)
    preds += model.predict(test_df, num_iteration=model.best_iteration)

# Average predictions
preds /= 10

# -------------------- Create Submission --------------------
submission = pd.DataFrame({
    'TransactionID': submission_ids,
    'isFraud': preds
})

submission.to_csv('IEEE-CIS_Submission/submission_full-dataset.csv', index=False)
print("Submission file saved as 'submission_lightgbm.csv'")


Loading model for Fold 1
Loading model for Fold 2
Loading model for Fold 3
Loading model for Fold 4
Loading model for Fold 5
Loading model for Fold 6
Loading model for Fold 7
Loading model for Fold 8
Loading model for Fold 9
Loading model for Fold 10
Submission file saved as 'submission_lightgbm.csv'
